## LeNet5 — Torch + LiteRT build pipeline

Trains a LeNet5 on MNIST, exports an int8 TFLite model via post-training quantization (analogous to the TensorFlow lenet notebook), using `litert-torch`.

Outputs

- `models/lenet5_quantized_torch.tflite` INT8 input/output, per-tensor, NHWC; MicroFlow-compatible

Keep `epochs` small if you only need artifacts quickly.

In [13]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

SEED = 3407
torch.manual_seed(SEED)

REPO_ROOT = Path.cwd().parent
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

OUT_TFLITE_QUANT = MODELS_DIR / "lenet5_quantized_torch.tflite"

DEVICE = "cpu"
print("Model output:", OUT_TFLITE_QUANT)
print("Device:", DEVICE)

Model output: /home/nathan/Documents/ariel-microflow-ml/models/lenet5_quantized_torch.tflite
Device: cpu


In [14]:
class LeNet5(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Conv2d(6, 16, kernel_size=5),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 120),
            nn.ReLU(),
            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Linear(84, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


transform = transforms.Compose([transforms.ToTensor()])

train_ds = datasets.MNIST(root=str(REPO_ROOT / "dataset"), train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root=str(REPO_ROOT / "dataset"), train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

model = LeNet5().to(DEVICE)
print(model)

LeNet5(
  (features): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
    (1): ReLU()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (4): ReLU()
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=256, out_features=120, bias=True)
    (2): ReLU()
    (3): Linear(in_features=120, out_features=84, bias=True)
    (4): ReLU()
    (5): Linear(in_features=84, out_features=10, bias=True)
  )
)


In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def evaluate(loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb).argmax(dim=1)
            correct += int((pred == yb).sum().item())
            total += yb.size(0)
    return correct / max(total, 1)


EPOCHS = 2
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in tqdm(train_loader, desc=f"epoch {epoch + 1}"):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    val_acc = evaluate(test_loader)
    print(f"epoch={epoch + 1} val_acc={val_acc:.4f}")

print("Test accuracy:", evaluate(test_loader))

epoch 1: 100%|██████████| 469/469 [00:05<00:00, 79.36it/s] 


epoch=1 val_acc=0.9402


epoch 2: 100%|██████████| 469/469 [00:05<00:00, 81.95it/s] 


epoch=2 val_acc=0.9713
Test accuracy: 0.9713


In [ ]:
import litert_torch
from ai_edge_litert import interpreter as tfl_interpreter
import tensorflow as tf

model.eval()

# Wrap model to accept NHWC
nhwc_model = litert_torch.to_channel_last_io(model, args=[0])
sample_input = torch.zeros(1, 28, 28, 1)  # NHWC

calib_loader = DataLoader(test_ds, batch_size=1, shuffle=False)


def representative_data_gen():
    for i, (xb, _) in enumerate(calib_loader):
        if i >= 200:
            break
        # dataloader gives NCHW; convert to NHWC for calibration
        yield [xb.permute(0, 2, 3, 1).numpy()]


tfl_flags = {
    "optimizations": [tf.lite.Optimize.DEFAULT],
    "representative_dataset": representative_data_gen,
    "target_spec": {
        "supported_ops": [tf.lite.OpsSet.TFLITE_BUILTINS_INT8],
        "supported_types": [tf.int8],
    },
    "inference_input_type": tf.int8,
    "inference_output_type": tf.int8,
    "_experimental_disable_per_channel": True,
    "_experimental_disable_per_channel_quantization": True,
}

edge_model = litert_torch.convert(
    nhwc_model, (sample_input,), _ai_edge_converter_flags=tfl_flags
)

edge_model.export(str(OUT_TFLITE_QUANT))
print("Wrote int8 per-tensor TFLite to", OUT_TFLITE_QUANT)

interp = tfl_interpreter.Interpreter(model_content=edge_model.tflite_model())
interp.allocate_tensors()
in_info = interp.get_input_details()[0]
out_info = interp.get_output_details()[0]
print("input: ", in_info["dtype"], in_info["shape"], "quant=", in_info["quantization"])
print("output:", out_info["dtype"], out_info["shape"], "quant=", out_info["quantization"])


INFO:tensorflow:Assets written to: /tmp/tmpr69lm9p8/assets


INFO:tensorflow:Assets written to: /tmp/tmpr69lm9p8/assets


Wrote int8 per-tensor TFLite to /home/nathan/Documents/ariel-microflow-ml/models/lenet5_quantized_torch.tflite
input:  <class 'numpy.int8'> [ 1 28 28  1] quant= (0.003921568859368563, -128)
output: <class 'numpy.int8'> [ 1 10] quant= (0.15269835293293, 27)


W0000 00:00:1776417808.305538   13337 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1776417808.305556   13337 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1776417808.305718   13337 reader.cc:83] Reading SavedModel from: /tmp/tmpr69lm9p8
I0000 00:00:1776417808.305998   13337 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1776417808.306003   13337 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpr69lm9p8
I0000 00:00:1776417808.308069   13337 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1776417808.321642   13337 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpr69lm9p8
I0000 00:00:1776417808.326244   13337 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 20535 microseconds.
I0000 00:00:1776417808.406348   13337 flatbuffer_export.cc:4160] Estimated count of arithmetic ops: 0.572 M  ops, equivalently 0.286 M  MACs
fully_quanti